# 6. Rekomendasi Keputusan
Menggabungkan hasil klasifikasi (risiko bulan depan), momentum inflasi, dan volatilitas klaster menjadi skor prioritas dan saran tindakan per pasar. Jalankan notebook 5 dan 6 lebih dulu.

In [ ]:
import re

import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, f1_score, silhouette_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.tree import DecisionTreeClassifier

import os
import matplotlib.pyplot as plt

klasifikasi_path = '../data/processed/hasil_klasifikasi_status.csv'
klaster_path = '../data/processed/klaster_pasar.csv'
output_dir = '../data/processed'

In [ ]:
KOL_PASAR = 'mkt_name'
KOL_TANGGAL = 'date'
KOL_INDEKS = 'food_price_index'
KOMODITAS = ['eggs', 'garlic', 'meat_beef', 'meat_chicken', 'meat_chicken_broiler', 'oil', 'onions', 'rice', 'sugar']

BATAS_WASPADA = 5
BATAS_LONJAKAN = 10
NAMA_KELAS = ['Normal', 'Waspada', 'Lonjakan']
FITUR_DASAR = ['inflasi_ini', 'inflasi_lalu', 'bulan']
PERSEN_TEST = 0.2
TAHUN_ACUAN_AKHIR = 2019

BOBOT_RISIKO = 0.6
BOBOT_MOMENTUM = 0.2
BOBOT_VOLATILITAS = 0.2
BATAS_PRIORITAS = 50
BATAS_PANTAU = 25
BATAS_HARGA_PERSEN = 5

NAMA_KOMODITAS = {
    'eggs': 'Telur', 'garlic': 'Bawang Putih', 'meat_beef': 'Daging Sapi', 'meat_chicken': 'Daging Ayam',
    'meat_chicken_broiler': 'Daging Ayam Broiler', 'oil': 'Minyak Goreng', 'onions': 'Bawang Merah',
    'rice': 'Beras', 'sugar': 'Gula', 'food_price_index': 'Indeks Pangan',
}
MUSIM_ONI = ['DJF', 'JFM', 'FMA', 'MAM', 'AMJ', 'MJJ', 'JJA', 'JAS', 'ASO', 'SON', 'OND', 'NDJ']

In [ ]:
def alasan_pasar(r):
    bagian = []
    peluang = r['p_waspada'] + r['p_lonjakan']
    if r['p_lonjakan'] >= 0.5:
        bagian.append(f"peluang lonjakan {r['p_lonjakan']:.0%}")
    elif peluang >= 0.5:
        bagian.append(f"peluang naik ke Waspada atau lebih {peluang:.0%}")
    if r['inflasi_ini'] - r['inflasi_lalu'] >= 1:
        bagian.append(f"inflasi naik dari {r['inflasi_lalu']:.1f}% ke {r['inflasi_ini']:.1f}%".replace('.', ','))
    if pd.notna(r['nilai_pemicu']) and r['nilai_pemicu'] >= BATAS_LONJAKAN:
        bagian.append(f"inflasi {r['komoditas_pemicu'].lower()} {r['nilai_pemicu']:.0f}%")
    if r['volatilitas_pct'] >= 67:
        bagian.append('harga sering bergejolak')
    teks = '; '.join(bagian[:3]) if bagian else 'kondisi stabil'
    if r['perlu_verifikasi']:
        teks += ' (perlu verifikasi data)'
    return teks[0].upper() + teks[1:]


def skor_rekomendasi(terbaru, prof):
    t = terbaru.copy()
    risiko = (t['p_waspada'] * 0.5 + t['p_lonjakan'] * 1.0) * 100
    momentum = ((t['inflasi_ini'] - t['inflasi_lalu']) / 10).clip(0, 1) * 100
    vol_pct = prof['volatilitas_inflasi'].rank(pct=True) * 100
    t['label_klaster'] = t[KOL_PASAR].map(prof['label_klaster'])
    t['volatilitas_pct'] = t[KOL_PASAR].map(vol_pct).fillna(vol_pct.median())
    t['skor_prioritas'] = (BOBOT_RISIKO * risiko + BOBOT_MOMENTUM * momentum + BOBOT_VOLATILITAS * t['volatilitas_pct']).round(1)
    t['tingkat'] = np.select([t['skor_prioritas'] >= BATAS_PRIORITAS, t['skor_prioritas'] >= BATAS_PANTAU],
                             ['Prioritas tinggi', 'Pantau ketat'], default='Aman')
    kol_inf = [c for c in t.columns if c.startswith('inf_')]
    if kol_inf:
        t['komoditas_pemicu'] = t[kol_inf].idxmax(axis=1).str.replace('inf_', '', regex=False).map(NAMA_KOMODITAS)
        t['nilai_pemicu'] = t[kol_inf].max(axis=1)
    else:
        t['komoditas_pemicu'] = 'Indeks Pangan'
        t['nilai_pemicu'] = t['inflasi_ini']
    aksi = {'Prioritas tinggi': 'Siapkan operasi pasar dan cek pasokan', 'Pantau ketat': 'Pantau harga mingguan', 'Aman': 'Pemantauan rutin bulanan'}
    t['rekomendasi'] = t['tingkat'].map(aksi)
    t.loc[t['tingkat'] != 'Aman', 'rekomendasi'] += ' (pemicu: ' + t.loc[t['tingkat'] != 'Aman', 'komoditas_pemicu'] + ')'
    tanpa_survei = t['obs_12b'] == 0 if 'obs_12b' in t.columns else pd.Series(False, index=t.index)
    rentang_lebar = t['rentang_harga'] >= t['rentang_harga'].quantile(0.9) if 'rentang_harga' in t.columns else pd.Series(False, index=t.index)
    t['perlu_verifikasi'] = tanpa_survei | rentang_lebar
    t['alasan'] = t.apply(alasan_pasar, axis=1)
    return t.sort_values('skor_prioritas', ascending=False)

## Baca hasil model
Skor = 0,6 x risiko (probabilitas Waspada dan Lonjakan) + 0,2 x momentum inflasi + 0,2 x persentil volatilitas klaster.

In [ ]:
terbaru = pd.read_csv(klasifikasi_path, parse_dates=[KOL_TANGGAL])
prof = pd.read_csv(klaster_path, index_col=0)
rek = skor_rekomendasi(terbaru, prof)
print(f'Bulan data: {terbaru[KOL_TANGGAL].max():%b %Y} | jumlah pasar: {len(rek)}')
rek['tingkat'].value_counts()

In [ ]:
rek[[KOL_PASAR, 'skor_prioritas', 'tingkat', 'status_prediksi_bulan_depan', 'label_klaster', 'alasan']].head(15)

## Visualisasi

In [ ]:
urut = ['Prioritas tinggi', 'Pantau ketat', 'Aman']
warna = {'Prioritas tinggi': '#c62828', 'Pantau ketat': '#f9a825', 'Aman': '#2e7d32'}
hitung = rek['tingkat'].value_counts().reindex(urut, fill_value=0)
hitung.plot(kind='bar', color=[warna[k] for k in urut], figsize=(6, 4), title='Jumlah pasar per tingkat prioritas')
plt.xticks(rotation=0)
plt.show()

In [ ]:
rek.head(15).set_index(KOL_PASAR)['skor_prioritas'].sort_values().plot(kind='barh', figsize=(8, 6), title='15 pasar dengan skor prioritas tertinggi')
plt.show()

## Simpan hasil

In [ ]:
kolom = [KOL_PASAR, 'skor_prioritas', 'tingkat', 'status_bulan_ini', 'status_prediksi_bulan_depan', 'komoditas_pemicu', 'label_klaster', 'alasan', 'perlu_verifikasi', 'rekomendasi']
rek[kolom].to_csv(f'{output_dir}/rekomendasi_keputusan.csv', index=False)
print('Rekomendasi tersimpan di', output_dir)